# Clustering depuis un fichier CSV de candidats doublons

L'objectif de ce Notebook est d'interpréter un fichier CSV qui définit des clusters d'Acteur et des créer les suggestions de clustering qui correspondent à ces clusters

Les 2 premières colonnes du fichier CSV sont l'identifiant du cluster et le uuid de l'acteur affiché
Les détails des acteurs sont récupérés à partir de la base de données, le dataframe des clusters est reconstitué avec le même format qu'à la sortir de l'étape `cluster_acteurs_clusters_prepare` du DAG de Clustering.

Chaque cluster du dataframe sont triés selon la liste de priorité des sources (établit selon leur fiabilité)

La logique métier des étapes suivantes du DAG de Custering est alors éxécuté :

- cluster_acteurs_parents_choose_new
- cluster_acteurs_parents_choose_data
- cluster_acteurs_suggestions_prepare
- cluster_acteurs_suggestions_to_db


## 1. Initialiser les variables d'environnement si besoin

In [9]:
import os

# Surdéfinition des variables d'environnement permettant d'accéder à la base de données
# car je n'ai pas réussi à préciser quel fichier d'env spécifique decouple doit
# interpréter
DATABASE_URL=''
DB_WAREHOUSE=''
# Si LOG_DF = True, On affiche des contenus de dataframe dans les logs
# Note : il fauttt éviter de commiter des résultats avec des LOG_DF=True
LOG_DF=False

if DATABASE_URL:
    os.environ["DATABASE_URL"] = DATABASE_URL
if DB_WAREHOUSE:
    os.environ["DB_WAREHOUSE"] = DB_WAREHOUSE


## 2. Initialiser l'environnement

- Charger les variables d'environnment à partir d'un fichier spécifique `webapp/.env` ou `webapp/.env.preprod`
- Chargement de Django et ajout des path dans sys.path


In [10]:
import sys
import os
from pathlib import Path

# Notebook : data-platform/notebooks
notebooks_dir = Path.cwd()
project_root = notebooks_dir.parent        # => data-platform

# On part du dossier du notebook: data-platform/notebooks
project_root = Path.cwd().parent   # => data-platform
sys.path.append(str(project_root))
sys.path.append(str(project_root / "dags"))

from dags.utils.django import django_setup_full

os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

django_setup_full()


## 3. Lecture du fichier CSV

Le fichier est déposé à coté du Notebook. \
Cette partie doit être adaptée selon le format du fichier fournit

In [ ]:
import pandas as pd

filepath = "candidats_doublons.csv"

# Lecture du CSV (séparateur ;, 2ème ligne = en-têtes)
df_raw = pd.read_csv(filepath, sep=";", header=1)
df_raw.columns = ["regroupement", "identifiant_unique", "url_admin", "nom", "adresse", "ville", "siret", "telephone"]
df_raw = df_raw[["regroupement", "identifiant_unique"]]
df_raw.rename(columns={"identifiant_unique": "uuid"}, inplace=True)

if LOG_DF:
    print(df_raw.head())


## 4. Construction du DataFrame `df_clusters` au format de sortie de `cluster_acteurs_clusters_prepare`

On récupère les données complètes des acteurs depuis la base, puis on trie par priorité de source dans chaque cluster.

In [12]:
import numpy as np
from qfdmo.models.acteur import Source, VueActeur, RevisionActeur

# Priorité des sources (par code) - les premières sont prioritaires
source_codes_priority = [
    "metropole_lille", "ecotempo", "repair_cafe", "tefal_seb", "larochelle",
    "cultura", "grandlyon", "mres_asso", "cma_reparacteur", "redeem_medical",
    "ecopae", "orange", "laposte", "ocad3e", "leroymerlin",
    "ordredespharmaciens", "lerefer", "villedeparis", "syvadec", "crarnormandie",
    "lunettesdezac", "emmausdefi", "recycloptics", "emmausconnect", "equipe",
    "carteco", "originecycles", "recyclivrepl", "alv", "soren",
    "pyreo", "cyclevia", "envie", "ressourceries", "findis",
    "communautelvao", "encore", "cmareparacteur", "lvao",
    "refashion_non_obligatoire", "lokki", "ocab_ecominero", "ocab_valdelia",
    "ocab_ecomaison", "ocab_valobat", "uniqueapp", "planeteliege",
    "deprecated_lvao", "valobat", "valdelia", "tyval", "batribox",
    "ecosystem", "ecominero", "ecologic", "ecodds", "dastri",
    "corepile", "citeo", "aliapur", "alcome", "adelphe",
    "ecomaison", "refashion", "ludotheques", "bibliotheques",
    "ademenationales", "sinoe", "ademelocales", "ademelocation",
    "reseau_vrac_reemploi",
]

# Convertir les codes en IDs de source
source_code_to_id = {}
source_id_priority = []
for code in source_codes_priority:
    try:
        source = Source.objects.get(code=code)
        source_code_to_id[code] = source.id
        source_id_priority.append(source.id)
    except Source.DoesNotExist:
        print(f"⚠️  Source '{code}' non trouvée en base, ignorée")

print(f"{len(source_id_priority)} sources trouvées sur {len(source_codes_priority)}")

# Colonnes à récupérer depuis VueActeur (comme dans cluster_acteurs_clusters_prepare)
fields_to_fetch = [
    "uuid", "identifiant_unique", "statut", "acteur_type__code", "code_postal",
    "ville", "location", "source__code", "nom", "action_principale_id",
    "source_id", "identifiant_externe", "acteur_type_id", "modifie_le",
    "cree_le", "est_parent", "nombre_enfants", "commentaires",
    "description", "adresse", "adresse_complement", "url", "email",
    "telephone", "nom_commercial", "nom_officiel", "siren", "siret",
    "naf_principal", "horaires_osm", "horaires_description",
    "public_accueilli", "reprise", "exclusivite_de_reprisereparation",
    "uniquement_sur_rdv", "consignes_dacces",
]

# Récupérer tous les identifiants uniques du CSV
all_ids = df_raw["uuid"].unique().tolist()
print(f"{len(all_ids)} acteurs uniques dans le CSV")

# Récupérer les données depuis la base
acteurs_qs = VueActeur.objects.filter(uuid__in=all_ids).values(*fields_to_fetch)
df_acteurs = pd.DataFrame(list(acteurs_qs)).replace({np.nan: None})

# Renommer les colonnes Django → noms attendus
df_acteurs = df_acteurs.rename(columns={
    "acteur_type__code": "acteur_type_code",
    "source__code": "source_code",
})

# Extraire latitude/longitude depuis le champ location (Point GIS)
if "location" in df_acteurs.columns:
    df_acteurs["latitude"] = df_acteurs["location"].apply(
        lambda loc: loc.y if loc else None
    )
    df_acteurs["longitude"] = df_acteurs["location"].apply(
        lambda loc: loc.x if loc else None
    )
    df_acteurs = df_acteurs.drop(columns=["location"])

acteurs_identifiant_uniques = list(set(df_acteurs["identifiant_unique"].tolist()))

# Récupérer les parent_id depuis RevisionActeur
acteur_to_parent_ids = dict(
    RevisionActeur.objects.filter(
        identifiant_unique__in=acteurs_identifiant_uniques, parent__isnull=False
    ).values_list("identifiant_unique", "parent__identifiant_unique")
)

df_acteurs["parent_id"] = df_acteurs["identifiant_unique"].map(
    lambda x: acteur_to_parent_ids.get(x, None)
)
print(f"acteur who have parent {len(df_acteurs[df_acteurs['parent_id'].notna()])}")

# Vérifier les acteurs manquants
ids_found = set(df_acteurs["uuid"].tolist())
ids_missing = set(all_ids) - ids_found
if ids_missing:
    print(f"⚠️  {len(ids_missing)} acteurs non trouvés en base: {ids_missing}")

print(f"{len(df_acteurs)} acteurs récupérés depuis la base")

if LOG_DF:
    print(df_acteurs.head())

# Construire df_clusters en associant chaque acteur à son cluster_id (regroupement du CSV)
# et en triant par priorité de source dans chaque cluster

# Créer le mapping identifiant_unique → regroupement
id_to_cluster = dict(zip(df_raw["uuid"], df_raw["regroupement"]))

# Joindre le cluster_id aux données complètes
df_clusters = df_acteurs.copy()
df_clusters["cluster_id"] = df_clusters["uuid"].map(id_to_cluster)

# Supprimer les acteurs sans cluster (ne devrait pas arriver)
df_clusters = df_clusters[df_clusters["cluster_id"].notna()]

# Fonction de tri par priorité de source dans chaque cluster
def source_sort_key(source_id):
    try:
        return source_id_priority.index(source_id)
    except ValueError:
        return float("inf")

df_clusters["_source_priority"] = df_clusters["source_id"].apply(source_sort_key)
df_clusters = df_clusters.sort_values(
    ["cluster_id", "_source_priority"]
).drop(columns=["_source_priority"]).reset_index(drop=True)

print(f"{len(df_clusters)} lignes, {df_clusters['cluster_id'].nunique()} clusters")
if LOG_DF:
    print("\nAperçu d'un cluster :")
    first_cluster = df_clusters["cluster_id"].iloc[800]
    df_clusters[df_clusters["cluster_id"] == first_cluster][
        ["uuid", "identifiant_unique", "nom", "source_code", "adresse", "ville", "est_parent", "parent_id"]
    ]


71 sources trouvées sur 71
1650 acteurs uniques dans le CSV
acteur who have parent 8
1650 acteurs récupérés depuis la base
1650 lignes, 815 clusters


## 5. Choix du parent pour chaque cluster (`cluster_acteurs_parents_choose_new`)

In [13]:
from cluster.tasks.business_logic.cluster_acteurs_parents_choose_new import (
    cluster_acteurs_parents_choose_new,
)

df_clusters = cluster_acteurs_parents_choose_new(df_clusters)

# Aperçu des décisions prises
from data.models.change import COL_CHANGE_MODEL_NAME

print("Répartition des types de changements :")
print(df_clusters[COL_CHANGE_MODEL_NAME].value_counts())
print(f"\n{len(df_clusters)} lignes, {df_clusters['cluster_id'].nunique()} clusters")

# Aperçu d'un cluster
if LOG_DF:
    first_cluster = df_clusters["cluster_id"].iloc[0]
    df_clusters[df_clusters["cluster_id"] == first_cluster][
        ["identifiant_unique", "nom", "source_code", COL_CHANGE_MODEL_NAME, COL_CHANGE_REASON, COL_CHANGE_ORDER, "parent_id"]
    ]

Répartition des types de changements :
change_model_name
acteur_update_parent_id               1593
acteur_create_as_parent                763
acteur_keep_as_parent                   52
acteur_verify_presence_in_revision       3
acteur_delete_as_parent                  2
Name: count, dtype: int64

2413 lignes, 815 clusters


## 6. Choix des données du parent selon les priorités de source (`cluster_acteurs_parents_choose_data`)

In [14]:
from cluster.tasks.business_logic.cluster_acteurs_parents_choose_data import (
    cluster_acteurs_parents_choose_data,
)
from cluster.config.constants import COL_PARENT_DATA_NEW

# Champs à inclure pour le choix des données du parent
fields_to_include = [
    "nom", "description", "adresse", "adresse_complement", "code_postal",
    "ville", "url", "email", "telephone", "nom_commercial", "nom_officiel",
    "siren", "siret", "naf_principal", "horaires_osm", "horaires_description",
    "public_accueilli", "reprise", "exclusivite_de_reprisereparation",
    "uniquement_sur_rdv", "consignes_dacces",
]

# Pas de sources exclues
exclude_source_ids = []

df_clusters = cluster_acteurs_parents_choose_data(
    df_clusters=df_clusters,
    fields_to_include=fields_to_include,
    exclude_source_ids=exclude_source_ids,
    prioritize_source_ids=source_id_priority,
    keep_empty=False,
    keep_parent_data_by_default=True,
)

# Aperçu des données choisies pour les parents
df_parents = df_clusters[df_clusters[COL_PARENT_DATA_NEW].notna()]
print(f"{len(df_parents)} parents avec des données à mettre à jour")

if LOG_DF:
    df_parents[["cluster_id", "identifiant_unique", COL_PARENT_DATA_NEW]].head()

815 parents avec des données à mettre à jour


## 7. Préparation des suggestions (`cluster_acteurs_suggestions_prepare`)

In [15]:
from cluster.tasks.business_logic.cluster_acteurs_suggestions.prepare import (
    cluster_acteurs_suggestions_prepare,
)
working, failing = cluster_acteurs_suggestions_prepare(df_clusters)

print(f"len working {len(working)}")
print(f"len failing {len(failing)}")

if LOG_DF:
    working[0]

len working 815
len failing 0


## 8. Ecriture des suggestions en DB (`cluster_acteurs_suggestions_to_db`)

In [16]:

from cluster.tasks.business_logic.cluster_acteurs_suggestions.to_db import (
    cluster_acteurs_suggestions_to_db,
)
from datetime import datetime

# Champs utilisés pour le contexte des suggestions
cluster_fields_exact = ["cluster_id"]
cluster_fields_fuzzy = []

identifiant_action = "notebook_clustering_from_file"
identifiant_execution = f"manual_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

cluster_acteurs_suggestions_to_db(
    df_clusters=df_clusters,
    suggestions=working,
    identifiant_action=identifiant_action,
    identifiant_execution=identifiant_execution,
    cluster_fields_exact=cluster_fields_exact,
    cluster_fields_fuzzy=cluster_fields_fuzzy,
)

print(f"✅ {len(working)} suggestions écrites en base")
print(f"Action: {identifiant_action}")
print(f"Exécution: {identifiant_execution}")

cluster_id='Regroupement 604' vide: bien rafraîchir les acteurs affichés
cluster_id='Regroupement 793' vide: bien rafraîchir les acteurs affichés


✅ 815 suggestions écrites en base
Action: notebook_clustering_from_file
Exécution: manual_20260311_155113
